# Run the trained LangGraph CLI agent

This notebook exercises the same validation and confirmation path as `python -m bash_agent.main_hf`. The local model emits the structured JSON used during training; an OpenAI-compatible model may instead make a native `run_langgraph` tool call. Both paths become the same validated, immutable argument vector.

## 1. Choose the fixed working directory

Set `LANGGRAPH_PROJECT_ROOT` to an existing LangGraph project for `dev`, `up`, `build`, or `dockerfile`. Point it at a trusted parent directory when asking the agent to create a new project. Generated paths cannot escape this root.

In [ ]:
import json
import os
from pathlib import Path

from bash_agent.bash import Bash
from bash_agent.config import Config
from bash_agent.main_hf import execute_with_confirmation, format_argv

root_value = os.getenv("LANGGRAPH_PROJECT_ROOT")
if not root_value:
    raise RuntimeError("Set LANGGRAPH_PROJECT_ROOT to a trusted existing directory.")
project_root = Path(root_value).resolve()
if not project_root.is_dir():
    raise NotADirectoryError(project_root)

pass_environment = {
    name.strip() for name in os.getenv("AGENT_PASS_ENV", "").split(",") if name.strip()
}
config = Config(
    root_dir=str(project_root),
    device=os.getenv("AGENT_DEVICE", "cuda"),
    pass_environment=pass_environment,
)
executor = Bash(config)
print(f"Fixed working directory: {executor.cwd}")

## 2. Inspect validation without executing anything

The structured training format is converted to canonical process arguments. It is never interpreted by a shell.

In [ ]:
sample_payload = {
    "command": "dockerfile",
    "output_path": "Dockerfile.langgraph",
}
sample_invocation = executor.prepare_payload(sample_payload)
print(format_argv(sample_invocation.argv))
assert sample_invocation.argv == ("langgraph", "dockerfile", "Dockerfile.langgraph")

In [ ]:
from bash_agent.commands import CommandValidationError

try:
    executor.prepare_argv(["sh", "-c", "echo unsafe"])
except CommandValidationError as error:
    print(f"Rejected as expected: {error}")

## 3. Load a local checkpoint or connect to an API

Set `AGENT_API_URL` and `AGENT_API_MODEL` to use a Nemotron model on a vLLM OpenAI-compatible endpoint. Set `AGENT_API_OMIT_THINKING_OVERRIDE=1` only for a strict endpoint that rejects vLLM's chat-template extension, and configure reasoning-off at that server. Otherwise the merged checkpoint from notebook 02 is loaded locally. Keep API credentials in `OPENAI_API_KEY`, not in notebook cells.

In [ ]:
from bash_agent.helpers import Messages, get_llm

api_url = os.getenv("AGENT_API_URL")
if api_url:
    config.use_api = True
    config.api_base_url = api_url
    config.api_model_name = os.environ["AGENT_API_MODEL"]
    config.api_send_thinking_override = os.getenv("AGENT_API_OMIT_THINKING_OVERRIDE") != "1"
else:
    model_path = Path(config.model_path)
    if not model_path.is_dir():
        raise FileNotFoundError(
            f"No merged checkpoint at {model_path}. Run notebook 02 or set AGENT_API_URL."
        )

llm = get_llm(config)

## 4. Ask for one tool call and validate it

Change the request to match the project root you selected. This cell only prepares and displays the invocation.

In [ ]:
messages = Messages(config.json_system_prompt)
messages.add_user_message(
    "Generate a LangGraph Dockerfile at the relative path Dockerfile.langgraph."
)
raw_response, tool_calls = llm.query(messages, [executor.to_json_schema()])
print(raw_response.split("</think>")[-1].strip())
if not tool_calls:
    raise RuntimeError("The model did not produce a valid LangGraph tool call.")

tool_call = tool_calls[0]
if hasattr(tool_call, "model_dump"):
    tool_call = tool_call.model_dump(exclude_none=True)
if not isinstance(tool_call, dict):
    raise TypeError("Tool call must serialize to an object.")
function = tool_call.get("function", {})
if function.get("name") != "run_langgraph":
    raise ValueError("Expected a run_langgraph tool call.")
arguments = json.loads(function["arguments"])
if not isinstance(arguments, dict) or set(arguments) != {"argv"}:
    raise ValueError("run_langgraph requires exactly one argv field.")
invocation = executor.prepare_argv(arguments["argv"])
print(f"Validated invocation: {format_argv(invocation.argv)}")

## 5. Optional execution with exact human confirmation

The next cell can write files, run project code, or invoke Docker depending on the proposed subcommand. Read the displayed arguments before answering. The runtime is not a sandbox. Common credentials are removed from the child environment unless their names are explicitly listed in `AGENT_PASS_ENV` (comma-separated).

In [ ]:
result = execute_with_confirmation(executor, invocation)
result

## Interactive launcher

For normal use, run the packaged loop from a terminal:

```bash
uv run --project training python -m bash_agent.main_hf \
  --model-path outputs/grpo_langgraph_cli/merged_model \
  --device cuda \
  --root-dir /path/to/trusted/langgraph-project
```

For a Nemotron model on a vLLM OpenAI-compatible server, add `--use-api --api-url URL --api-model MODEL` and set `OPENAI_API_KEY` in the environment. The client sends vLLM's reasoning-off chat-template extension by default. For a strict endpoint that rejects it, add `--omit-api-thinking-override` (or set `AGENT_API_OMIT_THINKING_OVERRIDE=1` in this notebook) and configure reasoning-off at the server. Use repeatable `--pass-env NAME` only when the reviewed child project needs a credential such as `LANGSMITH_API_KEY`. Finite commands use `--command-timeout` (600 seconds by default), and retained output/logs are capped at 1 MiB. Managed background processes stop and their temporary logs are deleted on normal or handled interactive exit; abrupt `SIGKILL` cannot run cleanup. Every tool call is revalidated immediately before execution, and every execution requires confirmation.

## Cleanup

The interactive launcher cleans up managed `dev` and non-waiting `up` process groups automatically. In the notebook, run the next cell when finished. Detached Docker services started by `up --wait` must be stopped with the normal Docker/LangGraph workflow.

In [ ]:
executor.close()